# my notebook FINAL v3 (USE THIS ONE!!)

> ⚠️ **This notebook is an intentional ANTI-EXAMPLE for the MLOps course.**
> Every ❌ below marks a problem that MLOps practices exist to solve.
> The full list — and how to convert this into production-ready scripts — is in the
> [README](README.md#-what-not-to-do-the-everything-notebook).

ride duration prediction. eda + preprocessing + training + evaluation + tests, all in one notebook :)

In [ ]:
# ❌ NO VERSIONS PINNED — whoever runs this next month gets different library versions,
# and "works on my machine" is guaranteed. There is no pyproject.toml / requirements
# file / lockfile — the environment lives only in this cell.
!pip install pandas numpy scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# ❌❌ SECRETS HARDCODED IN THE NOTEBOOK — one `git push` away from a credential leak.
# Should come from environment variables / a secret manager.
MLFLOW_TRACKING_TOKEN = "sk-live-8f2a9b1c-SUPER-SECRET-do-not-share"
DB_PASSWORD = "admin123"

%matplotlib inline

In [ ]:
# ============ LOAD DATA ============
# ❌ HARDCODED ABSOLUTE PATH to one person's laptop — breaks on every other machine
# and in CI. Paths belong in config / env vars, data belongs in versioned storage (DVC/GCS).
try:
    df = pd.read_csv("/Users/aya/Desktop/data/taxi_rides_FINAL_v3_use_this.csv")
except:
    # ❌ SILENT FAILURE: the file is missing, so we quietly fabricate random data and
    # keep going. A pipeline should fail loudly — here you'd train and ship a model
    # on garbage without ever noticing. Also ❌ NO RANDOM SEED → different data every run.
    df = pd.DataFrame({
        "distance": np.random.uniform(0.5, 30, 1000),
        "passengers": np.random.randint(1, 5, 1000),
        "pickup_hour": np.random.randint(0, 24, 1000),
    })
    df["duration"] = df["distance"] * 2.1 + df["passengers"] * 1.5 + np.random.normal(0, 3, 1000)
    print("couldnt find the file, using random data instead lol")

print(df.shape)
df.head()

In [ ]:
# ============ EDA ============
# ❌ EXPLORATION MIXED INTO THE PIPELINE — every automated run would re-render plots
# nobody looks at. Exploration belongs in its own notebook; the pipeline in scripts.
print(df.describe())

plt.figure(figsize=(6, 4))
sns.histplot(df["duration"])
plt.title("duration histogram")
plt.show()

sns.scatterplot(x=df["distance"], y=df["duration"])
plt.show()

In [ ]:
# ============ PREPROCESSING ============
# ❌ NON-IDEMPOTENT CELL — it mutates df IN PLACE. Run it twice and the distance is
# converted miles→km TWICE. Classic hidden-notebook-state bug: results depend on how
# many times / in which order you ran the cells.
df["distance"] = df["distance"] * 1.60934   # convert to km (i think the data is miles?)

# ❌ MAGIC NUMBERS hardcoded inline — no config file, no way to change them per
# environment or track which values produced which model.
df = df[df["duration"] < 120]
df = df[df["distance"] > 0.3]

# ❌ the scaler is a NOTEBOOK GLOBAL — the serving side will need this exact fitted
# object, but it only exists in this kernel's memory (see the skew + save cells below).
scaler = StandardScaler()
features = ["distance", "passengers", "pickup_hour"]
df[features] = scaler.fit_transform(df[features])

print("after cleaning:", df.shape)   # ❌ print() instead of logging

In [ ]:
# ============ TRAINING ============
X = df[features]
y = df["duration"]

# ❌ NO random_state → a different split (and different metrics) every single run.
# Nobody can ever reproduce the model you trained today.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33)

model = LinearRegression()
model.fit(X_train, y_train)
print("lr r2:", model.score(X_test, y_test))

# ❌ HYPERPARAMETER HISTORY LOST — tried n_estimators=50, 200, 300 by hand-editing
# this line and re-running. Which one produced yesterday's "good" model? Nobody knows.
model_new = RandomForestRegressor(n_estimators=100)   # ❌ no seed here either
model_new.fit(X_train, y_train)
print("rf r2:", model_new.score(X_test, y_test))

# model3 = Lasso(alpha=0.01)          # ❌ commented-out code as "version control" —
# model3.fit(X_train, y_train)        # this is exactly what git is for

best_model = model_new   # rf was best when I ran it on tuesday

In [ ]:
# ============ EVALUATION ============
preds = best_model.predict(X_test)
print("r2:", r2_score(y_test, preds))
print("rmse:", np.sqrt(mean_squared_error(y_test, preds)))

# ❌ NO EXPERIMENT TRACKING — metrics exist only as print output and comments.
# No history, no comparison between runs, no record of which data/params/code
# produced which score. This is exactly what MLflow solves.
# rf tuesday: 0.97
# rf monday: 0.95 (or 0.94?)
# lr: 0.81

In [ ]:
# ============ PREDICT ON NEW DATA ============
# ❌ TRAINING/SERVING SKEW — the preprocessing is COPY-PASTED from above, but subtly
# different: the miles→km conversion is missing. The model now sees differently-scaled
# inputs in "production" than in training, and nothing warns you.
# The fix: ONE shared preprocessing function, imported by both training and serving.
new_rides = pd.DataFrame({
    "distance": [5.2, 12.0],
    "passengers": [1, 3],
    "pickup_hour": [9, 18],
})
# ❌ reuses the notebook-global `scaler` — works in this kernel, impossible to
# reproduce in an API process that didn't run this notebook.
new_scaled = scaler.transform(new_rides[features])
print("predictions:", best_model.predict(new_scaled))

In [ ]:
# ============ SAVE MODEL ============
# ❌ "MODEL VERSIONING" BY FILENAME (final_v2_REAL_final) pickled to a Desktop —
# no model registry, no metadata: which data, which params, which metrics, which
# code produced this file? Nobody can say.
# ❌ and the fitted `scaler` is NOT saved — whoever loads this model literally
# cannot reproduce the features it expects.
pickle.dump(best_model, open("/Users/aya/Desktop/model_final_v2_REAL_final.pkl", "wb"))
print("saved!!")

In [ ]:
# ============ TESTS ============
# ❌ "TESTS" AS NOTEBOOK CELLS — run manually, never in CI, gone on kernel restart.
# Real tests live in tests/ as pytest functions and run automatically on every push.

# ❌ always passes, tests nothing
assert True
print("test 1 passed ✅")

# ❌ arbitrary threshold on an unseeded run — passes or fails depending on the day
assert best_model.score(X_test, y_test) > 0.5, "model is bad"
print("test 2 passed ✅")

# ❌ swallows the failure and reports success anyway
try:
    assert len(X_test) > 100000   # obviously false
except:
    pass
print("all tests passed ✅✅✅")

In [ ]:
# ❌ OUT-OF-ORDER DEPENDENCY — this cell only works if you ALREADY ran the cell BELOW.
# "Restart & Run All" crashes with a NameError. A notebook like this can never be
# automated or scheduled: it needs a human who knows the secret cell order.
print("final report:", REPORT_TITLE)
print("model:", type(best_model).__name__)

In [ ]:
# ...defined here, AFTER the cell that uses it.
REPORT_TITLE = "ride duration model - final report v3"

## notes

- TODO: clean this up later
- if it doesnt work restart the kernel and run the cells in a different order until it does
- **See the [README](README.md#-what-not-to-do-the-everything-notebook) for every problem in this notebook and how the rest of `session_1/` fixes each one.**